# Prompt 16: Handling Missing Data (NULLs)
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## NULL vs NaN — Critical Distinction

| | NULL | NaN |
|---|------|-----|
| Meaning | Absent / unknown value | "Not a Number" — result of invalid floating-point math |
| Type | Any column type | Floating-point columns only (`DOUBLE`, `FLOAT`) |
| Python | `None` | `float('nan')` |
| Spark check | `col('x').isNull()` | `F.isnan(col('x'))` |
| `isNull()` catches it? | ✅ | ❌ |
| `isnan()` catches it? | ❌ | ✅ |
| `dropna()` drops it? | ✅ | ❌ |
| `fillna()` fills it? | ✅ | ❌ — use `F.when(F.isnan(...), replacement)` |

**Exam rule:** `NaN != NULL`. They require different checks and different remediation.

## isNull() and isNotNull()

```python
from pyspark.sql.functions import col, isnan

df.filter(col('salary').isNull())          # rows where salary is NULL
df.filter(col('salary').isNotNull())       # rows where salary is NOT NULL
df.filter('salary IS NULL')               # SQL string equivalent
df.filter('salary IS NOT NULL')           # SQL string equivalent

# Check for NaN (floating-point only)
df.filter(isnan(col('salary')))
df.filter(F.isnan('salary'))              # string column name shortcut

# Combined: null OR NaN
df.filter(col('salary').isNull() | isnan(col('salary')))
```

## Counting NULLs

```python
# Count nulls in one column
df.filter(col('salary').isNull()).count()

# Count nulls per column — one-liner
df.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
]).show()

# Count nulls AND NaNs per column
df.select([
    F.count(F.when(col(c).isNull() | F.isnan(col(c)), 1)).alias(c)
    for c in df.columns
]).show()
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, isnan
import math

spark = SparkSession.builder \
    .appName('HandlingNulls') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

data = [
    (1,  'Alice',   'Engineering', 95000.0,        8,   True),
    (2,  'Bob',     'Marketing',   72000.0,         3,   True),
    (3,  'Carol',   'Engineering', None,            12,  False),
    (4,  'Dave',    None,          65000.0,         1,   True),
    (5,  'Eve',     'Engineering', float('nan'),    10,  True),
    (6,  'Frank',   'Marketing',   None,            None, True),
    (7,  'Grace',   'HR',          68000.0,         2,   False),
    (8,  'Henry',   'Engineering', 88000.0,         5,   True),
    (9,  None,      'Marketing',   79000.0,         6,   True),
    (10, 'Jack',    'Finance',     float('nan'),    7,   None),
]
schema = 'id INT, name STRING, department STRING, salary DOUBLE, years_exp INT, active BOOLEAN'
df = spark.createDataFrame(data, schema)
print('Dataset with NULLs and NaNs:')
df.show()
df.printSchema()

In [ ]:
# Cell 2: Checking for NULLs and NaNs

print('=== isNull() — finds NULL values ===')
df.filter(col('salary').isNull()).show()

print('=== isNotNull() ===')
df.filter(col('salary').isNotNull()).show()

print('=== isnan() — finds NaN values (floating-point only) ===')
df.filter(isnan(col('salary'))).show()   # Eve and Jack

print('=== isNull() does NOT catch NaN ===')
print('isNull count:', df.filter(col('salary').isNull()).count())    # Carol + Frank only
print('isnan count: ', df.filter(isnan(col('salary'))).count())     # Eve + Jack only

print('=== Combined check: NULL OR NaN ===')
df.filter(col('salary').isNull() | isnan(col('salary'))).show()
print('NULL or NaN count:', df.filter(col('salary').isNull() | isnan(col('salary'))).count())

print('=== Count NULLs per column ===')
df.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
]).show()

print('=== Count NULLs + NaNs per numeric column ===')
numeric_cols = ['salary']
df.select([
    F.count(F.when(col(c).isNull() | F.isnan(col(c)), 1)).alias(c + '_null_or_nan')
    for c in numeric_cols
]).show()

print('=== Null per column as percentage ===')
total = df.count()
df.select([
    F.round(F.count(F.when(col(c).isNull(), 1)) / total * 100, 1).alias(c)
    for c in df.columns
]).show()

## dropna() — Drop Rows with NULLs

```python
df.dropna()                            # drop rows with ANY null
df.na.drop()                           # identical — na is a DataFrameNaFunctions accessor

# how parameter
df.dropna(how='any')                   # drop if ANY column is null (default)
df.dropna(how='all')                   # drop only if ALL columns are null

# thresh parameter — minimum number of NON-null values to keep the row
df.dropna(thresh=4)                    # keep rows with at least 4 non-null values

# subset parameter — only check specific columns
df.dropna(subset=['salary', 'department'])   # drop if salary OR dept is null
df.dropna(how='any', subset=['salary'])      # drop if salary is null
df.dropna(how='all', subset=['salary', 'name'])  # drop if BOTH are null
```

**Key rule:** `dropna()` does **NOT** drop NaN. Use `filter(~isnan(...))` for NaN removal.

## fillna() — Replace NULLs with Values

```python
df.fillna(0)                           # fill ALL numeric nulls with 0
df.na.fill(0)                          # identical via na accessor

df.fillna('unknown')                   # fill ALL string nulls with 'unknown'
df.fillna(False)                       # fill ALL boolean nulls with False

# Fill specific columns with a dict
df.fillna({'salary': 0.0, 'department': 'Unknown', 'active': False})

# Fill specific columns with subset
df.fillna(0, subset=['salary', 'years_exp'])
df.fillna('Unknown', subset=['name', 'department'])
```

**Key rules:**
- `fillna()` only fills NULLs — **does NOT fill NaN**
- A scalar value is only applied to columns of a compatible type:
  - `fillna(0)` fills numeric columns only — ignores strings
  - `fillna('x')` fills string columns only — ignores numerics
- Use a **dict** to specify different fill values per column

## replace() — Replace Specific Values

```python
df.replace('old_val', 'new_val', subset=['col'])   # replace specific value
df.na.replace({'old1': 'new1', 'old2': 'new2'})   # dict replacement

# Replace can also handle nulls via dict with None key
df.replace(None, 'Unknown', subset=['department'])
```

In [ ]:
# Cell 3: dropna() — all parameters

print(f'Original row count: {df.count()}')

print('=== dropna() — drop rows with ANY null (default) ===')
df_any = df.dropna()
print(f'After dropna(): {df_any.count()} rows')
df_any.show()

print('=== dropna(how="all") — drop only if ALL values are null ===')
df_all = df.dropna(how='all')
print(f'After dropna(how="all"): {df_all.count()} rows (no all-null rows in data)')

print('=== dropna(thresh=N) — keep rows with at least N non-null values ===')
df_thresh = df.dropna(thresh=5)   # must have at least 5 non-null out of 6 columns
print(f'After dropna(thresh=5): {df_thresh.count()} rows')
df_thresh.show()

print('=== dropna(subset=[...]) — only check specific columns ===')
df_sub = df.dropna(subset=['salary'])
print(f'After dropna(subset=["salary"]): {df_sub.count()} rows')
df_sub.show()

print('=== dropna on multiple columns with how="all" ===')
df_sub_all = df.dropna(how='all', subset=['salary', 'department'])
print(f'After dropna(how="all", subset=["salary","department"]): {df_sub_all.count()} rows')

print('=== Important: dropna does NOT remove NaN rows ===')
# Eve (row 5) and Jack (row 10) have NaN salary — dropna won't remove them
after_drop = df.dropna(subset=['salary'])
print('After dropna on salary — NaN rows still present:')
after_drop.filter(isnan(col('salary'))).show()

print('=== Remove both NULL and NaN from salary ===')
df_clean = df.filter(col('salary').isNotNull() & ~isnan(col('salary')))
print(f'After removing NULL and NaN from salary: {df_clean.count()} rows')
df_clean.show()

In [ ]:
# Cell 4: fillna() and replace() — fill strategies

print('=== fillna(0) — fills all numeric nulls with 0 (ignores strings) ===')
df.fillna(0).show()

print('=== fillna("Unknown") — fills all string nulls (ignores numerics) ===')
df.fillna('Unknown').show()

print('=== fillna() with dict — different values per column ===')
df.fillna({
    'name':       'Unknown',
    'department': 'Unassigned',
    'salary':     0.0,
    'years_exp':  0,
    'active':     False,
}).show()

print('=== fillna() with subset ===')
df.fillna('MISSING', subset=['name', 'department']).show()
df.fillna(0, subset=['salary', 'years_exp']).show()

print('=== fillna does NOT fill NaN — only NULL ===')
df_filled = df.fillna(0)
print('After fillna(0) — NaN rows still have non-zero salary (NaN ≠ NULL):')
df_filled.filter(isnan(col('salary'))).show()

print('=== Handling NaN: use when() to replace NaN ===')
df_nan_fixed = df.withColumn(
    'salary',
    F.when(isnan(col('salary')), lit(None)).otherwise(col('salary'))
)
# Now NaN is converted to NULL, so fillna works
df_nan_fixed = df_nan_fixed.fillna({'salary': 0.0})
print('After NaN→NULL→fillna(0):')
df_nan_fixed.show()

print('=== replace() — replace specific values ===')
df.replace('Marketing', 'Sales', subset=['department']).show()
df.na.replace({'Marketing': 'Sales', 'HR': 'People'}, subset=['department']).show()

## coalesce() — Return First Non-Null from Multiple Columns

`F.coalesce()` evaluates expressions left to right and returns the first non-null value.
Equivalent to SQL `COALESCE(a, b, c)`.

```python
from pyspark.sql.functions import coalesce

# Return first non-null value from a prioritised list of columns
df.withColumn('email',
    coalesce(col('primary_email'), col('backup_email'), lit('no-email@company.com'))
)

# Null-safe fill from another column
df.withColumn('effective_salary',
    coalesce(col('override_salary'), col('base_salary'), lit(0.0))
)
```

**`coalesce()` vs `fillna()`:**
- `fillna(val)` — fill a column's NULLs with a **scalar** value
- `coalesce(colA, colB, lit(val))` — fill from **another column first**, then fallback

## NULL Behaviour in Aggregations — Critical Exam Facts

| Aggregation | Behaviour with NULLs |
|-------------|---------------------|
| `F.count('col')` | **Ignores NULLs** — only counts non-null values |
| `F.count('*')` | Counts ALL rows including nulls |
| `F.sum('col')` | **Ignores NULLs** — sums only non-null values |
| `F.avg('col')` | **Ignores NULLs** — averages only non-null values |
| `F.max('col')` | **Ignores NULLs** |
| `F.min('col')` | **Ignores NULLs** |
| `F.collect_list('col')` | **Includes NULLs** in the list |
| `F.collect_set('col')` | **Excludes NULLs** from the set |
| `F.countDistinct('col')` | **Ignores NULLs** |

**NULL in comparisons:**
```python
NULL == NULL  →  NULL  (not True!)
NULL != NULL  →  NULL
NULL > 5      →  NULL
# Use eqNullSafe() for null-safe equality:
col('a').eqNullSafe(col('b'))   # True when both are NULL
col('a').eqNullSafe(None)       # equivalent to isNull()
```

## NULL in Joins

```python
# NULL join keys are NEVER matched — standard join semantics
# Two rows with NULL key will NOT be joined to each other
df1.join(df2, on='key')          # rows with key=NULL in either side are dropped

# To join on null-safe equality:
df1.join(df2, df1['key'].eqNullSafe(df2['key']))
```

In [ ]:
# Cell 5: coalesce() and NULL behaviour in aggregations

print('=== coalesce() — first non-null from multiple sources ===')
# Create a DataFrame with two salary columns to demonstrate coalesce
data2 = [
    (1, 'Alice', 95000.0, None),
    (2, 'Bob',   None,    72000.0),
    (3, 'Carol', None,    None),
    (4, 'Dave',  88000.0, 91000.0),
]
df2 = spark.createDataFrame(data2, ['id', 'name', 'override_salary', 'base_salary'])

df2.withColumn(
    'effective_salary',
    F.coalesce(col('override_salary'), col('base_salary'), lit(0.0))
).show()

print('=== coalesce() in filter — keep rows where any salary source is present ===')
df2.filter(F.coalesce(col('override_salary'), col('base_salary')).isNotNull()).show()

print('=== NULL behaviour in aggregations ===')
print('count(col) ignores nulls, count(*) includes all rows:')
df.agg(
    F.count('salary').alias('count_salary'),    # ignores NULL
    F.count('*').alias('count_all'),             # all rows
    F.sum('salary').alias('sum_salary'),         # ignores NULL (and NaN!)
    F.avg('salary').alias('avg_salary'),         # ignores NULL
).show()

print('Difference: count(*) vs count(col)')
total = df.count()
non_null = df.filter(col('salary').isNotNull()).count()
null_count = df.filter(col('salary').isNull()).count()
print(f'count(*) = {total}, count(salary) = {non_null}, nulls = {null_count}')

print('=== collect_list includes NULLs, collect_set excludes them ===')
df.groupBy('department').agg(
    F.collect_list('salary').alias('salary_list'),
    F.collect_set('salary').alias('salary_set'),
).filter(col('department').isNotNull()).show(truncate=False)

print('=== NULL in comparisons — eqNullSafe ===')
# Standard equality: NULL == NULL is NULL (not True)
df.withColumn('salary_eq_null', col('salary') == lit(None)).show()

# eqNullSafe: NULL == NULL is True
df.withColumn('salary_null_safe', col('salary').eqNullSafe(None)).show()

In [ ]:
# Cell 6: NULL in sort, join, and window functions

print('=== NULL ordering in sort — asc puts NULLs first by default ===')
df.orderBy(col('salary').asc()).select('name', 'salary').show()
df.orderBy(col('salary').asc_nulls_last()).select('name', 'salary').show()
df.orderBy(col('salary').desc_nulls_last()).select('name', 'salary').show()

print('=== NULL in joins — NULL keys are never matched ===')
left  = spark.createDataFrame([(1,'Alice','Eng'),(2,None,'HR'),(3,'Carol','Fin')],
                               ['id','name','dept'])
right = spark.createDataFrame([(1,95000),(None,72000),(3,110000)],
                               ['id','salary'])

print('left DataFrame:')
left.show()
print('right DataFrame:')
right.show()

print('Inner join — NULL id rows from right are dropped:')
left.join(right, on='id', how='inner').show()   # row with id=None from right is dropped

print('Left join — NULL id from right side appears as null in result:')
left.join(right, on='id', how='left').show()

print('=== NULL-safe join with eqNullSafe ===')
left.join(right, left['id'].eqNullSafe(right['id']), how='inner').show()

print('=== NULL in window functions — nulls in partition/order keys ===')
from pyspark.sql.window import Window
w = Window.partitionBy('department').orderBy(col('salary').asc_nulls_last())
df.filter(col('department').isNotNull()) \
  .withColumn('rank', F.rank().over(w)) \
  .select('name', 'department', 'salary', 'rank') \
  .show()

In [ ]:
# Cell 7: Complete NULL handling pipeline — realistic data quality workflow

print('=== Full data quality pipeline ===')

# Step 1: Audit
print('--- Step 1: NULL/NaN audit ---')
total_rows = df.count()
null_audit = df.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])
print(f'Total rows: {total_rows}')
print('NULL counts per column:')
null_audit.show()

# Step 2: NaN → NULL normalisation
print('--- Step 2: Normalise NaN to NULL ---')
float_cols = [f.name for f in df.schema.fields if str(f.dataType) == 'DoubleType()']
print(f'Float columns to check for NaN: {float_cols}')
df_step2 = df
for c in float_cols:
    df_step2 = df_step2.withColumn(
        c,
        F.when(isnan(col(c)), lit(None)).otherwise(col(c))
    )
print('After NaN→NULL normalisation:')
df_step2.show()

# Step 3: Critical column drop (salary and name required)
print('--- Step 3: Drop rows missing critical columns ---')
df_step3 = df_step2.dropna(subset=['salary', 'name'])
print(f'After dropping rows with null salary or name: {df_step3.count()} rows')
df_step3.show()

# Step 4: Fill remaining nulls with defaults
print('--- Step 4: Fill remaining NULLs with defaults ---')
df_step4 = df_step3.fillna({
    'department': 'Unassigned',
    'years_exp':  0,
    'active':     False,
})
print('After fillna:')
df_step4.show()

# Step 5: Final audit
print('--- Step 5: Final NULL audit ---')
df_step4.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c)
    for c in df_step4.columns
]).show()
print(f'Final clean row count: {df_step4.count()}')

spark.stop()
print('Done.')

## Quick Reference Summary

### NULL detection
```python
col('x').isNull()                          # IS NULL
col('x').isNotNull()                       # IS NOT NULL
F.isnan(col('x'))                          # is NaN (float only)
col('x').isNull() | F.isnan(col('x'))      # NULL or NaN
col('a').eqNullSafe(col('b'))              # null-safe equality
col('a').eqNullSafe(None)                  # same as isNull()
```

### Count NULLs per column
```python
df.select([F.count(F.when(col(c).isNull(), 1)).alias(c) for c in df.columns])
```

### dropna
```python
df.dropna()                                # any null in any column
df.dropna(how='all')                       # only if ALL columns null
df.dropna(thresh=N)                        # keep if >= N non-null values
df.dropna(subset=['a', 'b'])               # check only cols a and b
# Note: dropna does NOT remove NaN
```

### fillna
```python
df.fillna(0)                               # all numeric nulls → 0
df.fillna('Unknown')                       # all string nulls → 'Unknown'
df.fillna({'col1': 0, 'col2': 'x'})       # per-column dict
df.fillna(0, subset=['a', 'b'])            # specific columns
# Note: fillna does NOT fill NaN
```

### NaN handling (float columns only)
```python
# Step 1: convert NaN to NULL
df.withColumn('x', F.when(F.isnan(col('x')), lit(None)).otherwise(col('x')))
# Step 2: now fillna works
.fillna({'x': 0.0})
```

### coalesce
```python
F.coalesce(col('a'), col('b'), lit(default))   # first non-null wins
```

### NULL in aggregations
```python
F.count('col')     # ignores NULLs
F.count('*')       # counts all including NULLs
F.sum / avg / max / min   # all ignore NULLs
F.collect_list     # INCLUDES NULLs
F.collect_set      # EXCLUDES NULLs
```

## Real-World Scenarios

<details>
<summary>Scenario 1: IoT sensor data — handle missing readings with coalesce and fillna</summary>

**Situation:**
IoT sensors occasionally fail to report. The pipeline has a primary and a backup sensor.
If both are null, use the last known value from a separate lookup table; otherwise default to -999.

**Code:**
```python
readings_with_fallback = sensor_readings \
    .join(last_known.select('device_id', 'last_temp'), on='device_id', how='left') \
    .withColumn(
        'temperature',
        F.coalesce(
            col('primary_temp'),
            col('backup_temp'),
            col('last_temp'),
            lit(-999.0)
        )
    ) \
    .withColumn('imputed', col('primary_temp').isNull())
```

**Exam Sub-topic:** `coalesce()` chain; adding an `imputed` flag with `isNull()`
</details>

<details>
<summary>Scenario 2: ETL pipeline — classify rows by NULL severity before writing</summary>

**Situation:**
A pipeline routes records with critical nulls to a quarantine table, fills optional nulls,
and writes clean records to the main table.

**Code:**
```python
CRITICAL = ['order_id', 'customer_id', 'amount']
OPTIONAL = ['promo_code', 'notes', 'secondary_email']

# Build NOT-NULL condition dynamically
from functools import reduce
critical_ok = reduce(lambda a, b: a & b, [col(c).isNotNull() for c in CRITICAL])

clean      = df.filter(critical_ok) \
               .fillna({c: '' for c in OPTIONAL})

quarantine = df.filter(~critical_ok) \
               .withColumn('quarantine_reason',
                   F.concat_ws(', ', *[
                       F.when(col(c).isNull(), lit(c)) for c in CRITICAL
                   ]))

clean.write.mode('append').delta('/warehouse/orders')
quarantine.write.mode('append').delta('/quarantine/orders')
```

**Exam Sub-topic:** Dynamic compound `isNotNull()` filter; `fillna()` with dict; `when()` to build reason string
</details>

<details>
<summary>Scenario 3: Financial data — NaN from divide-by-zero, normalise before aggregation</summary>

**Situation:**
Calculating profit margin `(revenue - cost) / revenue` can produce NaN when `revenue == 0`.
NaN values must be normalised before aggregation to avoid misleading averages.

**Code:**
```python
# Calculate margin — may produce NaN when revenue is 0
df = df.withColumn('margin', (col('revenue') - col('cost')) / col('revenue'))

# Normalise: NaN → NULL, then fill with 0
df = df.withColumn(
    'margin',
    F.when(F.isnan(col('margin')) | col('margin').isNull(), lit(0.0))
     .otherwise(col('margin'))
)

# Now avg() gives correct result (no NaN contamination)
df.groupBy('region').agg(F.avg('margin').alias('avg_margin')).show()
```

**Exam Sub-topic:** NaN from arithmetic; combined `isnan() | isNull()` check; normalise before aggregation
</details>

<details>
<summary>Scenario 4: Survey data — distinguish "not answered" (NULL) from "answered 0" (zero)</summary>

**Situation:**
Survey responses use NULL for skipped questions and 0 for "strongly disagree".
Averages must only count questions that were answered (non-null).

**Code:**
```python
# F.avg() already ignores NULLs — so unanswered questions are automatically excluded
survey_stats = responses.groupBy('survey_id').agg(
    F.avg('q1_score').alias('q1_avg'),   # ignores null (skipped)
    F.avg('q2_score').alias('q2_avg'),
    F.count('q1_score').alias('q1_answered'),  # non-null count
    F.count('*').alias('total_respondents'),
)

# Response rate per question
survey_stats = survey_stats.withColumn(
    'q1_response_rate',
    F.round(col('q1_answered') / col('total_respondents') * 100, 1)
)
```

**Exam Sub-topic:** `F.avg()` ignores NULL; `F.count('col')` vs `F.count('*')` for non-null vs total
</details>

<details>
<summary>Scenario 5: Deduplication with null keys — safe matching with eqNullSafe</summary>

**Situation:**
A customer master table has some rows with NULL `external_id`.
A CDC feed needs to be matched including NULL-to-NULL matching for records without external IDs.

**Code:**
```python
# Standard join: NULL keys never match
bad_join = master.join(cdc, on='external_id')   # misses NULL-keyed rows

# Null-safe join: NULL == NULL is treated as True
safe_join = master.join(
    cdc,
    master['external_id'].eqNullSafe(cdc['external_id']),
    how='inner'
)

# Count the difference
print(f'Standard join rows:   {bad_join.count()}')
print(f'Null-safe join rows:  {safe_join.count()}')
```

**Exam Sub-topic:** NULL join key semantics; `eqNullSafe()` for null-safe equality in joins
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| NULL vs NaN | NULL = absent value; NaN = invalid float math — **completely different** |
| `isNull()` catches NaN? | ❌ No |
| `isnan()` catches NULL? | ❌ No |
| Correct NaN check | `F.isnan(col('x'))` — float columns only |
| Combined check | `col('x').isNull() \| F.isnan(col('x'))` |
| `dropna()` drops NaN? | ❌ No — only drops NULL rows |
| `fillna()` fills NaN? | ❌ No — only fills NULL |
| NaN → NULL pattern | `F.when(F.isnan(col('x')), lit(None)).otherwise(col('x'))` |
| `dropna()` default | `how='any'` — drops row if ANY column is null |
| `dropna(how='all')` | Drops row only if ALL columns are null |
| `dropna(thresh=N)` | Keep rows with at least N non-null values |
| `dropna(subset=[...])` | Only check specified columns |
| `fillna(0)` | Fills numeric NULLs only |
| `fillna('x')` | Fills string NULLs only |
| `fillna({'col': val})` | Different fill values per column |
| `F.coalesce(a, b, c)` | Returns first non-null from left to right |
| `F.count('col')` | **Ignores NULLs** |
| `F.count('*')` | Counts ALL rows including nulls |
| `F.sum/avg/max/min` | All **ignore NULLs** |
| `F.collect_list` | **Includes** NULLs |
| `F.collect_set` | **Excludes** NULLs |
| NULL in comparisons | `NULL == NULL` → NULL (not True!) |
| Null-safe equality | `col('a').eqNullSafe(col('b'))` — NULL==NULL → True |
| NULL join keys | Standard joins **never match** NULL keys |
| NULL in sort asc | NULLs appear **first** by default |
| Override NULL sort | `.asc_nulls_last()` / `.desc_nulls_last()` |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between NULL and NaN in Spark?</summary>

**A:**
- **NULL** — represents an absent or unknown value. Can appear in any column type (string, int, double, boolean). Detected with `.isNull()` / `.isNotNull()`. Handled by `dropna()`, `fillna()`, `coalesce()`.
- **NaN** — "Not a Number" — a special floating-point value resulting from invalid arithmetic (e.g., `0.0 / 0.0`, `sqrt(-1)`). Only appears in `DOUBLE` or `FLOAT` columns. Detected with `F.isnan()`. **NOT** handled by `dropna()` or `fillna()` — must be converted to NULL first using `F.when(F.isnan(...), None).otherwise(...)`.

They are completely independent — a cell can be NULL, NaN, or a valid number, and you must check each separately.
</details>

<details>
<summary>Q2: How does fillna() behave with mixed-type DataFrames?</summary>

**A:** `fillna(scalar)` only fills columns of a **compatible type**:
- `df.fillna(0)` — fills NULL in **numeric** columns (INT, LONG, DOUBLE) with 0. String and boolean columns are **unchanged**.
- `df.fillna('Unknown')` — fills NULL in **string** columns. Numeric columns are unchanged.
- `df.fillna(False)` — fills NULL in **boolean** columns only.

To fill different columns with different values in one call, use a **dict**: `df.fillna({'col1': 0, 'col2': 'Unknown', 'col3': False})`
</details>

<details>
<summary>Q3: Does F.count('col') count NULL values?</summary>

**A:** No. `F.count('col')` counts only **non-null** values. To count all rows including nulls, use `F.count('*')` or `F.count(lit(1))`.

Example:
- If a column has 10 rows but 3 are NULL: `F.count('col')` = 7, `F.count('*')` = 10.
- The difference `F.count('*') - F.count('col')` = number of NULLs in that column.

All other aggregate functions (`sum`, `avg`, `max`, `min`, `stddev`) also ignore NULLs.
</details>

<details>
<summary>Q4: What is coalesce() and how does it differ from fillna()?</summary>

**A:**
- `F.coalesce(col('a'), col('b'), lit(0.0))` — returns the **first non-null** value from the given expressions, evaluated left to right. Useful for combining multiple source columns with a fallback.
- `df.fillna(0)` — fills all NULL values in a column with a **scalar constant**. Cannot fall back to another column's value.

Use `coalesce()` when you need: "use column A; if null, try column B; if null, use 0".
Use `fillna()` when you just need: "replace all nulls in this column with a constant".
</details>

<details>
<summary>Q5: How does Spark handle NULL values in joins?</summary>

**A:** In standard join semantics, **NULL is never equal to anything**, including another NULL. This means:
- In an `inner` or `left` join on a key column: rows where the join key is NULL on either side are **dropped** (inner) or produce NULLs (left outer).
- Two rows both with `key = NULL` will **not** be matched together.

To join on NULL-safe equality (NULL == NULL → True), use `eqNullSafe()`:
```python
df1.join(df2, df1['key'].eqNullSafe(df2['key']), how='inner')
```
</details>